# Effect of pressure on flux and apparent permeability

Temperature is held at `CONDITIONS['T_operating']` and the upstream pressure is swept over `CONDITIONS['P_range']` (both from `study_config.py`).

The flux follows $J = A\,P^{\,n}$, so each fit is a straight line on log-log axes and its slope $n$ names the controlling step: $n \approx 1$ for a Henry oxide (Model 1) or the surface rate (Model 2), $n = 0.5$ for Sieverts transport.

$\Phi_{app} = J\,L/\Delta\sqrt{P}$ is computed from the same flux. Because $\Delta\sqrt{P}$ moves with the x-axis, its slope is $n - 0.5$: **flat where Sieverts transport controls** (the plateau is a true permeability) and rising as $\sqrt{P}$ where it does not. For L2a the normalisation is $\Delta P$, so its slope is $n - 1$.

## 1. Setup

In [ ]:
import sys; sys.path.insert(0, '..')
import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt
from scipy import stats
from calculations.config.model_config import (CONDITIONS, MICROSTRUCTURE,
                                              OXIDE_DEFECTS, build_simulation_config)
from calculations.oxide_permeation import (get_oxide_properties_at_T, get_metal_properties_at_T,
                                           molecular_diffusion_flux)
from calculations.permeation_calc import calculate_simple_metal_flux, calculate_defective_metal_flux
from calculations.interface_solver import calculate_oxide_metal_system
from calculations.parallel_oxide_defect_paths import (calculate_parallel_path_flux,
                                                      calculate_parallel_path_flux_defective_metal)
from calculations.wall_labels import label_model1, label_model2
from calculations.permeability import UNITS_SIEVERTS, UNITS_HENRY
import pandas as pd
from IPython.display import display
from calculations.figures import results_dir, save_figure
FIGS = results_dir("effect-of-pressure")                             # figures and fit tables go in this folder

SIM          = build_simulation_config()
T_op, P_down = CONDITIONS['T_operating'], SIM['P_downstream']
L_ox, L_m    = SIM['L_oxide'], SIM['L_metal']
P = np.logspace(*np.log10(CONDITIONS['P_range']), CONDITIONS['n_points'])   # upstream pressures (Pa)
P = P[P > P_down]                                                            # driving force must be positive

# Temperature is fixed, so the layer properties are computed once
OX = get_oxide_properties_at_T(SIM['oxide_name'], T_op); OX['thickness'] = L_ox
ME = get_metal_properties_at_T(SIM['metal_name'], T_op); ME['thickness'] = L_m

LEVELS = {   # level: (title, wall thickness for Φ_app)
    'L1':  ('Level 1: Perfect metal (lattice diffusion)',          L_m),
    'L2a': ('Level 2a: Perfect oxide alone (molecular H$_2$)',     L_ox),
    'L2b': ('Level 2b: Perfect oxide + perfect metal',             L_ox + L_m),
    'L3':  ('Level 3: Defective oxide + perfect metal',            L_ox + L_m),
    'L4':  ('Level 4: Defective metal (traps + grain boundaries)', L_m),
    'L5':  ('Level 5: Defective oxide + defective metal',          L_ox + L_m),
}

## 2. Physics (one solver call per level, then the sweep and the fit)

In [ ]:
def solve(level, Pk):
    """Flux and regime label for one level at one upstream pressure (T fixed at T_op)."""
    if level.endswith('L6'):
        return solve_m2(level, Pk)                       # Model 2 levels, defined further down
    ox, me = dict(OX), dict(ME)
    D, K  = me['D_metal'], me['K_s_metal']
    micro = dict(microstructure_params=MICROSTRUCTURE, lattice_density=MICROSTRUCTURE['N_L'], mode='both')
    if level == 'L1':  return calculate_simple_metal_flux(D, K, L_m, Pk, P_down)['flux'], label_model1('L1')
    if level == 'L2a': return molecular_diffusion_flux(ox['D_ox'], ox['K_ox'], L_ox, Pk, P_down), label_model1('L2a')
    if level == 'L2b': r = calculate_oxide_metal_system(Pk, P_down, ox, me)
    if level == 'L3':  r = calculate_parallel_path_flux(Pk, P_down, ox, me, OXIDE_DEFECTS)
    if level == 'L4':  r = calculate_defective_metal_flux(D_lattice=D, K_s=K, metal_thickness=L_m,
                                                          P_up=Pk, P_down=P_down, temperature=T_op, **micro)
    if level == 'L5':  r = calculate_parallel_path_flux_defective_metal(Pk, P_down, ox, me,
                                                                        OXIDE_DEFECTS, T_op, **micro)
    return r.get('flux_total', r.get('flux')), label_model1(level, r, Pk, P_down, ox, me, OXIDE_DEFECTS, T_op, micro)

def sweep(level):
    # Φ_app written out: apparent_permeability() takes one pressure, not an array of them
    J, regime = map(np.array, zip(*[solve(level, Pk) for Pk in P]))
    if level == 'L2a':                                   # Henry: J·L/ΔP
        return J, J * L_ox / (P - P_down), UNITS_HENRY, regime
    return J, J * LEVELS[level][1] / (np.sqrt(P) - np.sqrt(P_down)), UNITS_SIEVERTS, regime

def power_law(P, y):
    """y = A·P^n, fitted as ln y = ln A + n·ln P."""
    fit = stats.linregress(np.log(P), np.log(y))
    return fit.slope, np.exp(fit.intercept), fit.rvalue**2   # n, A, R²

## 3. Style

In [ ]:
mpl.rcParams.update({
    'font.size': 14, 'font.weight': 'bold', 'mathtext.default': 'bf',
    'axes.labelweight': 'bold', 'axes.titleweight': 'bold', 'axes.linewidth': 1.8,
    'xtick.direction': 'in', 'ytick.direction': 'in', 'ytick.right': True,
    'xtick.major.width': 1.6, 'ytick.major.width': 1.6,
    'legend.frameon': False, 'legend.fontsize': 11, 'savefig.dpi': 600, 'savefig.bbox': 'tight',
})
INK, GREY, SHADE = '#0b0b0b', '#52514e', '#e1e0d9'

# Regime labels name the dominant path and the limiting step inside it (calculations/wall_labels.py):
# colour = limiting step, hollow = a defect path carries most of the flux, diamond = trap-controlled.
from calculations.wall_labels import describe as look, fitted
UNITS_TEX = {UNITS_SIEVERTS: r'mol m$^{-1}$ s$^{-1}$ Pa$^{-1/2}$',
             UNITS_HENRY:    r'mol m$^{-1}$ s$^{-1}$ Pa$^{-1}$'}

## 4. Plot

In [ ]:
from matplotlib.lines import Line2D

LINES, ROLES = ['-', '--', ':'], ['low', 'mid', 'high']   # fit line style per condition, in order

def p_tag(Pc):
    """Short pressure label: 1e-07 -> '1e-7 Pa', 1.0 -> '1e0 Pa', 1e+07 -> '1e7 Pa'."""
    return f'{Pc:.0e}'.replace('e+0', 'e').replace('e-0', 'e-') + ' Pa'

def framed(nrows, figsize, sharex=True):
    """Figure with two legend strips under the title and an nrows x 2 grid of panels below them."""
    fig  = plt.figure(figsize=figsize, layout='constrained')
    gs   = fig.add_gridspec(nrows + 2, 2, height_ratios=[0.001, 0.001] + [1] * nrows)
    keys = [fig.add_subplot(gs[k, :]) for k in (0, 1)]
    for k in keys:
        k.axis('off')
    ax = np.empty((nrows, 2), dtype=object)
    for r in range(nrows):
        for c in range(2):
            ax[r, c] = fig.add_subplot(gs[r + 2, c], sharex=ax[0, c] if (r and sharex) else None)
    return fig, keys, ax

def shared_legend(keys, axes, lines):
    """One legend per figure: the markers (by regime) on the first strip, the lines on the second."""
    marks = {}
    for a in axes:
        for h, label in zip(*a.get_legend_handles_labels()):
            marks.setdefault(label, h)
    keys[0].legend(list(marks.values()), list(marks), title='Simulation', loc='center', ncol=min(len(marks), 2))
    keys[1].legend(handles=lines, loc='center', ncol=len(lines))

def condition_lines(tags, fit_label):
    """Line key: one style per condition, or a single fit line when there is one condition."""
    if len(tags) == 1:
        return [Line2D([], [], color=INK, lw=2.4, label=fit_label)]
    return [Line2D([], [], color=INK, lw=2.4, ls=LINES[ci], label=f'{ROLES[ci]}: {tag}')
            for ci, tag in enumerate(tags)]

def end_label(ax, xv, y, tag, ci):
    """Name a curve at its right-hand end, staggered so coinciding curves stay readable."""
    ok = np.flatnonzero(np.isfinite(y))
    i = ok[np.argmax(xv[ok])]
    ax.annotate(tag, (xv[i], y[i]), xytext=(12, 14 * (ci - 1)), textcoords='offset points',
                va='center', fontsize=11, color=INK)

def fit_table(rows):
    """Every fitted line as one row: shown under the figure, and returned for the CSV."""
    table = pd.DataFrame(rows).replace({r'\$_2\$': '₂'}, regex=True)
    with pd.option_context('display.float_format', '{:.4g}'.format, 'display.width', 220,
                           'display.max_columns', 20):
        display(table)
    return table

EXPONENT = 'n (J) / slope (Φ_app)'                  # Φ_app ∝ P^(n − 0.5), or P^(n − 1) for L2a

def slope_guides(ax, y):
    """Thin grey reference lines for n = 1 (steeper) and n = 0.5, through the middle simulation point."""
    i = len(P) // 2
    for n in (1.0, 0.5):
        ax.loglog(P, y[i] * (P / P[i])**n, '-', color=GREY, lw=1.0, alpha=0.6, zorder=0)

def panel(ax, y, regime, by_regime, rows, quantity, tag, ls):
    """Simulation points plus power-law fit(s), one group or one per regime; each fit becomes a table row."""
    for reg in (dict.fromkeys(regime) if by_regime else ['all']):
        m = (regime == reg) if by_regime else np.full(len(P), True)
        name, c, mk, *fill = look(reg) if by_regime else ('All pressures (single fit)', GREY, 'o')
        hollow = fill == ['hollow']                      # Model 2: flux mainly through oxide defects
        ax.loglog(P[m], y[m], mk, ms=9, color=c, mfc='white' if hollow else c,
                  mec=c if hollow else 'white', mew=2 if hollow else 1, label=name)
        if not fitted(reg) or m.sum() < 3:
            continue                                     # drawn, not fitted
        n, A, r2 = power_law(P[m], y[m])
        ax.loglog(P[m], A * P[m]**n, ls, lw=2.8, color=c if by_regime else INK)
        rows.append({'quantity': quantity, 'condition': tag, 'fit': name if by_regime else 'single fit',
                     'P window (Pa)': f'{P[m].min():.1e}–{P[m].max():.1e}', 'points': int(m.sum()),
                     EXPONENT: n, 'A (SI)': A, 'R²': r2})
    ax.grid(color=SHADE)

def pressure_figure(level, curves=None, note=None):
    """2x2 log-log figure and its fit table. curves = [(tag, sweep output), ...], one per condition."""
    title, L = LEVELS[level]
    curves = curves or [(f'{T_op:.0f} K', sweep(level))]
    multi, units = len(curves) > 1, curves[0][1][2]
    quantities = [('J',       r'Flux $J$ (mol m$^{-2}$ s$^{-1}$)'),
                  ('Phi_app', rf'$\Phi_{{app}}$ ({UNITS_TEX[units]})')]
    fig, keys, ax = framed(2, (14, 13))
    rows = []
    for r, (qty, ylab) in enumerate(quantities):
        ys = np.concatenate([out[r] for _, out in curves])
        for col in (0, 1):
            p = ax[r, col]
            if r == 0:
                slope_guides(p, curves[len(curves) // 2][1][0])
            for ci, (tag, out) in enumerate(curves):    # out = (J, Φ_app, units, regime)
                panel(p, out[r], out[3], col == 1, rows, qty, tag, LINES[ci] if multi else '-')
                if multi:
                    end_label(p, P, out[r], tag, ci)
            p.set_ylim(np.nanmin(ys) / 3, np.nanmax(ys) * 3)          # keep the guides from stretching the axis
            p.set_xlim(P.min() / 3, P.max() * (300 if multi else 3))   # room for the end labels
            p.set_ylabel(ylab)
    for p in ax[0]:
        p.tick_params(labelbottom=False)
    for p in ax[1]:
        p.set_xlabel(r'Upstream pressure $P_{up}$ (Pa)')
    ax[0, 0].set_title('Single power-law fit (all pressures)')
    ax[0, 1].set_title('Regime-resolved power-law fits')
    lines = condition_lines([t for t, _ in curves], 'Power-law fit (black: single, colour: per regime)')
    lines.append(Line2D([], [], color=GREY, lw=1.0, alpha=0.6, label='Reference slopes $n$ = 1 and 0.5'))
    shared_legend(keys, ax.flat, lines)
    drive = r'\Delta P' if units == UNITS_HENRY else r'\Delta\sqrt{P}'
    where = note or rf'$T$ = {T_op:.0f} K'
    fig.suptitle(f'{title}\n{where},   ' + rf'$J \propto P^n$,   $\Phi_{{app}} = J\,L/{drive}$,   $L$ = {L:.2e} m',
                 fontsize=18, fontweight='bold')
    plt.show()
    return fig, fit_table(rows)

## 5. Model 1 (L1–L5)

In [ ]:
for level in LEVELS:
    fig, fits = pressure_figure(level)
    save_figure(fig, f"{level}_pressure", FIGS)
    fits.to_csv(FIGS / f"{level}_pressure_fits.csv", index=False)

## 6. Model 2 (L6 family)

In [ ]:
# =============================================================================
# MODEL 2 (L6 family): dissociation on the gas-facing surface, atomic H in oxide and metal
# Reuses solve() -> sweep() -> pressure_figure() above; only the solver is new.
# =============================================================================
from calculations.surface_kinetics import (get_all_properties, solve_steady_state_flux_L1L6,
                                           solve_steady_state_flux_L2aL6, solve_steady_state_flux_direct,
                                           calculate_mixed_defect_flux_L6, calculate_defective_metal_flux_L6,
                                           calculate_full_model_flux_L346_v2)
from calculations.config.model_config import METALS, get_surface_kinetics_from_config

MICRO   = SIM['microstructure']
DEFECTS = SIM['defect_config']
SIEVERTS_PINHOLE = SIM['oxide_defects'].get('use_sieverts_pinhole', False)

# Temperature is fixed, so the Model 2 properties are computed once
PROPS = get_all_properties(SIM['oxide_name'], SIM['metal_name'], T_op)       # oxide surface + transport
SM    = get_surface_kinetics_from_config(SIM['metal_name'], T_op, METALS)    # bare-metal surface

LEVELS_M2 = {   # level: (title, wall thickness for Φ_app) -- every Model 2 level is Sieverts
    'L1L6':  ('Level 1+6: Perfect metal + surface kinetics',                     L_m),
    'L2aL6': ('Level 2a+6: Perfect oxide alone + surface kinetics',              L_ox),
    'L2bL6': ('Level 2b+6: Perfect oxide + perfect metal + surface kinetics',    L_ox + L_m),
    'L3L6':  ('Level 3+6: Defective oxide + perfect metal + surface kinetics',   L_ox + L_m),
    'L4L6':  ('Level 4+6: Perfect oxide + defective metal + surface kinetics',   L_ox + L_m),
    'L5L6':  ('Level 5+6: Defective oxide + defective metal + surface kinetics', L_ox + L_m),
}
LEVELS.update(LEVELS_M2)

def solve_m2(level, Pk):
    """Flux and regime label for one Model 2 level at one pressure, e.g. 'oxide surface' or 'pinholes/metal diffusion'."""
    k, Keq, Dox, Kox, Dm, Ksm = (PROPS[key] for key in ('k_diss', 'K_eq', 'D_ox', 'K_ox', 'D_m', 'K_s_m'))
    pin  = {} if SIEVERTS_PINHOLE else dict(k_diss_metal=SM['k_diss'], K_eq_metal=SM['K_eq'])
    mode = MICRO.get('mode', 'both')
    if level == 'L1L6':  r = solve_steady_state_flux_L1L6(Pk, P_down, L_m, SM['k_diss'], SM['K_eq'], Dm, Ksm)
    if level == 'L2aL6': r = solve_steady_state_flux_L2aL6(Pk, P_down, k, Keq, Dox, Kox, L_ox)
    if level == 'L2bL6': r = solve_steady_state_flux_direct(Pk, P_down, L_m, k, Keq, Dox, Kox, L_ox, Dm, Ksm)
    if level == 'L3L6':  r = calculate_mixed_defect_flux_L6(Pk, P_down, L_m, k, Keq, Dox, Kox, L_ox,
                                                            Dm, Ksm, DEFECTS, **pin)
    if level == 'L4L6':  r = calculate_defective_metal_flux_L6(Pk, P_down, L_m, T_op, k, Keq, Dox, Kox, L_ox,
                                                               Dm, Ksm, microstructure_params=MICRO, mode=mode)
    if level == 'L5L6':  r = calculate_full_model_flux_L346_v2(Pk, P_down, L_m, T_op, k, Keq, Dox, Kox, L_ox,
                                                               Dm, Ksm, MICRO, DEFECTS, mode=mode, **pin)
    return r.get('J_total', r.get('J_ss', r.get('flux'))), label_model2(level, r, DEFECTS)

for level in LEVELS_M2:
    fig, fits = pressure_figure(level)
    save_figure(fig, f"{level}_pressure", FIGS)
    fits.to_csv(FIGS / f"{level}_pressure_fits.csv", index=False)

## 7. Three temperatures: the pressure sweep at the low / mid / high of `T_range`

In [ ]:
# =============================================================================
# THREE TEMPERATURES: the pressure sweep at the low / mid / high of T_range
# Same figure as above, with one curve per temperature in each panel. Colour and marker still
# mean the regime; line style and the label at the end of each curve mean the temperature.
# =============================================================================
from contextlib import contextmanager

T_lo, T_hi = CONDITIONS['T_range']
T_SET = [T_lo, (T_lo + T_hi) / 2, T_hi]             # low / mid / high: 573, 923, 1273 K

@contextmanager
def at_temperature(Tc):
    """Run solve() and sweep() above at another temperature; T_op and its properties are restored afterwards."""
    global T_op, OX, ME, PROPS, SM
    saved = T_op, OX, ME, PROPS, SM
    T_op  = Tc
    OX    = dict(get_oxide_properties_at_T(SIM['oxide_name'], Tc), thickness=L_ox)
    ME    = dict(get_metal_properties_at_T(SIM['metal_name'], Tc), thickness=L_m)
    PROPS = get_all_properties(SIM['oxide_name'], SIM['metal_name'], Tc)
    SM    = get_surface_kinetics_from_config(SIM['metal_name'], Tc, METALS)
    try:
        yield
    finally:
        T_op, OX, ME, PROPS, SM = saved

for level in LEVELS:
    curves = []
    for Tc in T_SET:
        with at_temperature(Tc):
            curves.append((f'{Tc:.0f} K', sweep(level)))
    fig, fits = pressure_figure(level, curves, note='three temperatures (low / mid / high of T_range)')
    save_figure(fig, f"{level}_pressure_3T", FIGS)
    fits.to_csv(FIGS / f"{level}_pressure_3T_fits.csv", index=False)